# NS Work
### A harness for Northstar

## Setup
Configure here. Submit requests in NS Work.


In [ ]:
import os
from dotenv import load_dotenv
from company import ROOT

load_dotenv(ROOT / ".env", override=True)


## Our starting point: Deep Agents
Configure the loop for Northstar.

In [ ]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5.4",
    api_key=os.environ["OPENAI_API_KEY"],
    reasoning_effort="low",
    max_tokens=6000,
    use_responses_api=True,
    timeout=90,
    max_retries=1,
)

harness = create_deep_agent(model=model)


### Update the UI
Interface code: `server.py` · `web/`

In [ ]:
import interface
from importlib import reload
reload(interface)
from interface import update_ui

update_ui("base")


## Four parts of our harness

## 1. Context
What does it need to know?

### 1A. Company instructions

In [ ]:
from IPython.display import Markdown, display

display(Markdown((ROOT / "company/AGENTS.md").read_text()))

### 1B. Shared skills
Reusable procedures, loaded when needed.

In [ ]:
display(Markdown((ROOT / "skills/company/write-proposal/SKILL.md").read_text()))
display(Markdown((ROOT / "skills/team/triage-issue/SKILL.md").read_text()))

### Harness with Context

In [ ]:
from workspace import load_references

# Make company instructions and skills available for reading.
references = load_references()

harness = create_deep_agent(
    model=model,
    **references,
    system_prompt="Return drafts and findings in chat.",
    memory=["/company/AGENTS.md"],
    skills=["/skills/company/", "/skills/team/"],
)

### Update the UI

In [ ]:
update_ui("context")


## 2. Actions
What can it do?

### 2A. Company tools
[Customer lookup](tools/customer.py) · [Issue lookup](tools/issues.py) · [Revenue](tools/revenue.py) · [Proposal sending](tools/proposals.py)

In [ ]:
from company import Company

company = Company((ROOT / ".runtime/company.sqlite").resolve())

# Connect tools to company data and the employee role.
employee_role = "sales"  # Current employee. The UI has a separate Profile selector.
company_tools = company.all_tools_for(employee_role)
customer_brief = company_tools["customer_brief"]
issue_lookup = company_tools["issue_lookup"]
revenue_summary = company_tools["revenue_summary"]
send_proposal = company_tools["send_proposal"]


### 2B. Planning
`TodoListMiddleware` tracks steps. `ModelCallLimitMiddleware` bounds the run.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, TodoListMiddleware

agent_middleware = [
    TodoListMiddleware(),
    ModelCallLimitMiddleware(run_limit=10, exit_behavior="end"),
]


### 2C. Subagents
The proposal reviewer gets its own instructions, model and customer lookup.

In [ ]:
def proposal_reviewer(model, lookup):
    """Review customer evidence without access to proposal sending."""
    return {
        "name": "proposal-reviewer",
        "description": "Review a proposal against customer facts; flag unsupported commitments.",
        "system_prompt": "Look up the customer. Return evidence, assumptions and corrections.",
        "model": model,
        "tools": [lookup],
        "middleware": [ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")],
    }

reviewer_model = model
reviewer = proposal_reviewer(reviewer_model, customer_brief)


### Harness with Actions

In [ ]:
harness = create_deep_agent(
    model=model,
    **references,
    system_prompt="Return drafts and findings in chat.",
    memory=["/company/AGENTS.md"],
    skills=["/skills/company/", "/skills/team/"],
    tools=[
        customer_brief,
        issue_lookup,
        revenue_summary,
        send_proposal,
    ],
    subagents=[reviewer],
    middleware=agent_middleware,
)


### Update the UI

In [ ]:
update_ui("actions")


## 3. Boundaries
What is it allowed to do?

### 3A. Permissions

In [ ]:
# Define which company tools each employee role may use.
company.access = {
    "sales": {"customer_brief", "send_proposal"},
    "engineering": {"customer_brief", "issue_lookup"},
    "finance": {"customer_brief", "revenue_summary"},
}

allowed_tools = company.tools_for(employee_role)


In [ ]:
from deepagents.middleware.filesystem import FilesystemPermission

file_permissions = [
    FilesystemPermission(
        operations=["write"],
        paths=["/company/**", "/skills/**"],
        mode="deny",
    ),
]


### 3B. Approvals
Pause sending for human review.

### Harness with Boundaries

In [ ]:
harness = create_deep_agent(
    model=model,
    **references,
    system_prompt="Return drafts and findings in chat.",
    memory=["/company/AGENTS.md"],
    skills=["/skills/company/", "/skills/team/"],
    tools=allowed_tools,
    subagents=[reviewer],
    permissions=file_permissions,
    interrupt_on={"send_proposal": {"allowed_decisions": ["approve", "reject"]}},
    middleware=agent_middleware,
)


### Update the UI

In [ ]:
update_ui("boundaries")


## 4. Persistence
How does it keep work going?

### 4A. Working files

Connect a writable folder for proposals and other working files.


In [ ]:
from deepagents.backends import FilesystemBackend
from workspace import prepare_workspace

workspace = prepare_workspace(ROOT / "workspace" / "notebook")
backend = FilesystemBackend(root_dir=workspace, virtual_mode=True)


### 4B. Checkpoints
Same thread, saved state.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

checkpoint_path = ROOT / "checkpoints/notebook.sqlite"
checkpoint_path.parent.mkdir(exist_ok=True)

connection = sqlite3.connect(checkpoint_path, check_same_thread=False)
saver = SqliteSaver(connection)


### 4C. Context management
Summarize at 12,000 tokens and retain six recent messages.

In [ ]:
from deepagents.middleware.summarization import SummarizationMiddleware

agent_middleware.extend([
    SummarizationMiddleware(
        model=model,
        backend=backend,
        trigger=("tokens", 12000),
        keep=("messages", 6),
        trim_tokens_to_summarize=None,
    ),
])


### Harness with Persistence


In [ ]:
harness = create_deep_agent(
    model=model,
    memory=["/company/AGENTS.md"],
    skills=["/skills/company/", "/skills/team/"],
    tools=allowed_tools,
    subagents=[reviewer],
    permissions=file_permissions,
    interrupt_on={"send_proposal": {"allowed_decisions": ["approve", "reject"]}},
    middleware=agent_middleware,
    backend=backend,
    checkpointer=saver,
)


### Update the UI

In [ ]:
update_ui("persistence")
